In [1]:
import torch
import numpy as np
from lightning import pytorch as pl
from chemprop import data, models, nn

# SMILES string to predict (DSTOG, disodium tectorigenin)
SMILES_LIST = [
    "COc1c(O[Na])cc2occ(-c3ccc(O[Na])cc3)c(=O)c2c1O",
]

CHECKPOINT_PATH = "multitask_classification_model.cpt"
TARGET_NAMES = ["TRAF6", "NFKB1", "NFKBIA", "MAP3K7", "JUN", "IRAK1"]

torch.set_float32_matmul_precision('high')

In [2]:
# Model architecture must match the training configuration (multi_tasknn.py)
message_passing = nn.BondMessagePassing(d_h=500, depth=3, dropout=0.1)
agg = nn.MeanAggregation()
ffn = nn.BinaryClassificationFFN(
    n_tasks=len(TARGET_NAMES),
    input_dim=message_passing.output_dim,
    hidden_dim=300,
    n_layers=2,
    dropout=0.1
)

model = models.MPNN.load_from_checkpoint(
    CHECKPOINT_PATH,
    message_passing=message_passing,
    agg=agg,
    predictor=ffn,
    metrics=[],
    map_location="cuda"
)
model.eval()

test_data = [data.MoleculeDatapoint.from_smi(s) for s in SMILES_LIST]
test_dataset = data.MoleculeDataset(test_data)
test_loader = data.build_dataloader(test_dataset, batch_size=64, shuffle=False)

trainer = pl.Trainer(accelerator="gpu", devices=1, precision=32, logger=False, enable_progress_bar=False)
with torch.no_grad():
    preds_raw = trainer.predict(model, test_loader)
    all_preds = torch.cat(preds_raw, dim=0)
    probs = torch.sigmoid(all_preds).cpu().numpy()

print("\n" + "=" * 65)
print(f"{'SMILES':<30} | {'Target':<8} | {'Prob':<8} | {'Prediction'}")
print("-" * 65)

for i, smi in enumerate(SMILES_LIST):
    for j, target in enumerate(TARGET_NAMES):
        prob = probs[i][j]
        label = "Active" if prob > 0.5 else "Inactive"
        display_smi = (smi[:27] + '...') if len(smi) > 30 else smi
        print(f"{display_smi:<30} | {target:<8} | {prob:.4f} | {label}")
    print("-" * 65)

SMILES                         | Target   | Prob     | Prediction
-----------------------------------------------------------------
COc1c(O[Na])cc2occ(-c3ccc(O... | TRAF6    | 0.5958 | Active
COc1c(O[Na])cc2occ(-c3ccc(O... | NFKB1    | 0.5000 | Active
COc1c(O[Na])cc2occ(-c3ccc(O... | NFKBIA   | 0.5587 | Active
COc1c(O[Na])cc2occ(-c3ccc(O... | MAP3K7   | 0.5575 | Active
COc1c(O[Na])cc2occ(-c3ccc(O... | JUN      | 0.5577 | Active
COc1c(O[Na])cc2occ(-c3ccc(O... | IRAK1    | 0.5056 | Active
-----------------------------------------------------------------

